In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from IPython.display import display

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_DIR = Path("../")
RESULTS_DIR = PROJECT_DIR / "results"

print("Project:", PROJECT_DIR.resolve())
print("Results:", RESULTS_DIR.resolve())

Project: C:\Users\ASUS\Documents\Smart-Tourism-Recommendation
Results: C:\Users\ASUS\Documents\Smart-Tourism-Recommendation\results


In [3]:
recommendations_path = (
    RESULTS_DIR / "demand_aware_recommendations.csv"
)

recommendations = pd.read_csv(
    recommendations_path
)

print("Recommendations loaded successfully.")
print("Shape:", recommendations.shape)

print("\nColumns:")
print(recommendations.columns.tolist())

Recommendations loaded successfully.
Shape: (500, 17)

Columns:
['user_id', 'destination_id', 'name', 'category', 'district', 'estimated_cost_lkr', 'duration_hours', 'sustainability_score', 'crowd_score', 'predicted_crowd_score', 'predicted_crowd_level', 'seasonality_score', 'sbert_hybrid_score', 'crowd_suitability', 'demand_aware_score', 'demand_aware_rank', 'demand_explanation']


In [4]:
display(recommendations.head(10))

,user_id,destination_id,name,category,district,estimated_cost_lkr,duration_hours,sustainability_score,crowd_score,predicted_crowd_score,predicted_crowd_level,seasonality_score,sbert_hybrid_score,crowd_suitability,demand_aware_score,demand_aware_rank,demand_explanation
0,U001,D048,Colombo National Museum,heritage;culture,colombo,3000,2.5,58,26,29.084064,Low,53,0.538383,0.871676,0.638371,1,Recommended based on semantic preference match...
1,U001,D051,Viharamahadevi Park,nature;city,colombo,2000,2.0,58,29,32.059847,Low,56,0.618589,0.606397,0.614931,2,Recommended based on semantic preference match...
2,U001,D029,Ella Rock,nature;adventure,badulla,5000,6.0,61,29,30.727841,Low,59,0.530365,0.725140,0.588797,3,Recommended based on semantic preference match...
3,U001,D054,Mount Lavinia Beach,beach,colombo,5000,3.0,59,37,31.827333,Low,62,0.528928,0.627124,0.558387,4,Recommended based on semantic preference match...
4,U001,D053,Dutch Hospital Colombo,heritage;city,colombo,5000,2.5,60,30,33.808425,Medium,63,0.589435,0.450518,0.547760,5,Recommended based on semantic preference match...
5,U001,D020,Nuwara Eliya City,city;nature,nuwara eliya,5000,3.0,67,34,33.082968,Medium,57,0.527826,0.515189,0.524035,6,Recommended based on semantic preference match...
6,U001,D066,Bopath Ella,nature,ratnapura,3000,3.0,57,34,33.713491,Medium,56,0.541864,0.458981,0.516999,7,Recommended based on semantic preference match...
7,U001,D098,Ohiya,nature;adventure,badulla,5000,4.0,63,38,35.161483,Medium,66,0.557703,0.329898,0.489361,8,Recommended based on semantic preference match...
8,U001,D070,Brief Garden,nature,kalutara,4000,2.5,59,37,35.603315,Medium,67,0.532182,0.290510,0.459681,9,Recommended based on semantic preference match...
9,U001,D052,Independence Square,heritage;city,colombo,2000,2.0,60,38,36.897572,Medium,49,0.552463,0.175132,0.439264,10,Recommended based on semantic preference match...


In [5]:
print("Users:", recommendations["user_id"].nunique())
print("Rows:", len(recommendations))

print("\nRank range:")
print(
    recommendations["demand_aware_rank"].min(),
    "to",
    recommendations["demand_aware_rank"].max()
)

Users: 50
Rows: 500

Rank range:
1 to 10


In [6]:
required_columns = [
    "user_id",
    "destination_id",
    "name",
    "category",
    "estimated_cost_lkr",
    "sustainability_score",
    "predicted_crowd_score",
    "predicted_crowd_level",
    "weather_suitability",
    "sbert_hybrid_score",
    "crowd_suitability",
    "demand_aware_score",
    "demand_aware_rank"
]

print("Column availability:")

for col in required_columns:
    print(
        f"{col}:",
        col in recommendations.columns
    )

Column availability:
user_id: True
destination_id: True
name: True
category: True
estimated_cost_lkr: True
sustainability_score: True
predicted_crowd_score: True
predicted_crowd_level: True
weather_suitability: False
sbert_hybrid_score: True
crowd_suitability: True
demand_aware_score: True
demand_aware_rank: True


In [7]:
explanation_columns = [
    "user_id",
    "destination_id",
    "name",
    "category",
    "estimated_cost_lkr",
    "sustainability_score",
    "predicted_crowd_score",
    "predicted_crowd_level",
    "sbert_hybrid_score",
    "crowd_suitability",
    "demand_aware_score",
    "demand_aware_rank"
]

available_columns = [
    col for col in explanation_columns
    if col in recommendations.columns
]

print("Available explanation columns:")
print(available_columns)

explanation_data = recommendations[
    available_columns
].copy()

print("\nShape:", explanation_data.shape)

Available explanation columns:
['user_id', 'destination_id', 'name', 'category', 'estimated_cost_lkr', 'sustainability_score', 'predicted_crowd_score', 'predicted_crowd_level', 'sbert_hybrid_score', 'crowd_suitability', 'demand_aware_score', 'demand_aware_rank']

Shape: (500, 12)


In [8]:
print("Missing values:")

display(
    explanation_data.isna().sum()
)

Missing values:


user_id                  0
destination_id           0
name                     0
category                 0
estimated_cost_lkr       0
sustainability_score     0
predicted_crowd_score    0
predicted_crowd_level    0
sbert_hybrid_score       0
crowd_suitability        0
demand_aware_score       0
demand_aware_rank        0
dtype: int64

In [9]:
def minmax_normalize(series):
    min_value = series.min()
    max_value = series.max()

    if max_value == min_value:
        return pd.Series(
            1.0,
            index=series.index
        )

    return (
        (series - min_value)
        / (max_value - min_value)
    )


explanation_data["semantic_strength"] = (
    minmax_normalize(
        explanation_data["sbert_hybrid_score"]
    )
)

explanation_data["sustainability_strength"] = (
    minmax_normalize(
        explanation_data["sustainability_score"]
    )
)

explanation_data["crowd_strength"] = (
    explanation_data["crowd_suitability"]
)

print("Normalized explanation components created.")

display(
    explanation_data[
        [
            "name",
            "semantic_strength",
            "sustainability_strength",
            "crowd_strength"
        ]
    ].head(10)
)

Normalized explanation components created.


,name,semantic_strength,sustainability_strength,crowd_strength
0,Colombo National Museum,0.401955,0.333333,0.871676
1,Viharamahadevi Park,0.582132,0.333333,0.606397
2,Ella Rock,0.383943,0.476190,0.725140
3,Mount Lavinia Beach,0.380717,0.380952,0.627124
4,Dutch Hospital Colombo,0.516640,0.428571,0.450518
5,Nuwara Eliya City,0.378240,0.761905,0.515189
6,Bopath Ella,0.409776,0.285714,0.458981
7,Ohiya,0.445355,0.571429,0.329898
8,Brief Garden,0.388026,0.380952,0.290510
9,Independence Square,0.433586,0.428571,0.175132


In [10]:
def generate_explanation(row):

    reasons = []

    # Semantic recommendation
    if row["semantic_strength"] >= 0.70:
        reasons.append(
            "strong match with your travel preferences"
        )
    elif row["semantic_strength"] >= 0.40:
        reasons.append(
            "good match with your travel preferences"
        )
    else:
        reasons.append(
            "some alignment with your travel preferences"
        )

    # Crowd
    crowd_level = str(
        row["predicted_crowd_level"]
    ).lower()

    if crowd_level == "low":
        reasons.append(
            "predicted low crowd conditions"
        )
    elif crowd_level == "medium":
        reasons.append(
            "predicted moderate crowd conditions"
        )
    elif crowd_level == "high":
        reasons.append(
            "predicted high crowd conditions"
        )

    # Sustainability
    if row["sustainability_strength"] >= 0.70:
        reasons.append(
            "good sustainability characteristics"
        )

    return "Recommended because of " + ", ".join(
        reasons
    ) + "."


explanation_data["explanation"] = (
    explanation_data.apply(
        generate_explanation,
        axis=1
    )
)

display(
    explanation_data[
        [
            "user_id",
            "name",
            "demand_aware_rank",
            "explanation"
        ]
    ].head(10)
)

,user_id,name,demand_aware_rank,explanation
0,U001,Colombo National Museum,1,Recommended because of good match with your tr...
1,U001,Viharamahadevi Park,2,Recommended because of good match with your tr...
2,U001,Ella Rock,3,Recommended because of some alignment with you...
3,U001,Mount Lavinia Beach,4,Recommended because of some alignment with you...
4,U001,Dutch Hospital Colombo,5,Recommended because of good match with your tr...
5,U001,Nuwara Eliya City,6,Recommended because of some alignment with you...
6,U001,Bopath Ella,7,Recommended because of good match with your tr...
7,U001,Ohiya,8,Recommended because of good match with your tr...
8,U001,Brief Garden,9,Recommended because of some alignment with you...
9,U001,Independence Square,10,Recommended because of good match with your tr...


In [11]:
def generate_detailed_explanation(row):

    return (
        f"{row['name']} was recommended because it "
        f"has a semantic preference score of "
        f"{row['sbert_hybrid_score']:.3f}, "
        f"a crowd suitability score of "
        f"{row['crowd_suitability']:.3f}, "
        f"and a sustainability score of "
        f"{row['sustainability_score']:.1f}. "
        f"The predicted crowd level is "
        f"{row['predicted_crowd_level']}."
    )


explanation_data["detailed_explanation"] = (
    explanation_data.apply(
        generate_detailed_explanation,
        axis=1
    )
)

display(
    explanation_data[
        [
            "name",
            "demand_aware_score",
            "detailed_explanation"
        ]
    ].head(10)
)

,name,demand_aware_score,detailed_explanation
0,Colombo National Museum,0.638371,Colombo National Museum was recommended becaus...
1,Viharamahadevi Park,0.614931,Viharamahadevi Park was recommended because it...
2,Ella Rock,0.588797,Ella Rock was recommended because it has a sem...
3,Mount Lavinia Beach,0.558387,Mount Lavinia Beach was recommended because it...
4,Dutch Hospital Colombo,0.547760,Dutch Hospital Colombo was recommended because...
5,Nuwara Eliya City,0.524035,Nuwara Eliya City was recommended because it h...
6,Bopath Ella,0.516999,Bopath Ella was recommended because it has a s...
7,Ohiya,0.489361,Ohiya was recommended because it has a semanti...
8,Brief Garden,0.459681,Brief Garden was recommended because it has a ...
9,Independence Square,0.439264,Independence Square was recommended because it...


In [12]:
final_explainable = explanation_data[
    [
        "user_id",
        "destination_id",
        "name",
        "category",
        "estimated_cost_lkr",
        "sustainability_score",
        "predicted_crowd_score",
        "predicted_crowd_level",
        "sbert_hybrid_score",
        "crowd_suitability",
        "demand_aware_score",
        "demand_aware_rank",
        "explanation",
        "detailed_explanation"
    ]
].copy()

print("Final explainable recommendations created.")
print("Shape:", final_explainable.shape)

display(
    final_explainable.head(10)
)

Final explainable recommendations created.
Shape: (500, 14)


,user_id,destination_id,name,category,estimated_cost_lkr,sustainability_score,predicted_crowd_score,predicted_crowd_level,sbert_hybrid_score,crowd_suitability,demand_aware_score,demand_aware_rank,explanation,detailed_explanation
0,U001,D048,Colombo National Museum,heritage;culture,3000,58,29.084064,Low,0.538383,0.871676,0.638371,1,Recommended because of good match with your tr...,Colombo National Museum was recommended becaus...
1,U001,D051,Viharamahadevi Park,nature;city,2000,58,32.059847,Low,0.618589,0.606397,0.614931,2,Recommended because of good match with your tr...,Viharamahadevi Park was recommended because it...
2,U001,D029,Ella Rock,nature;adventure,5000,61,30.727841,Low,0.530365,0.725140,0.588797,3,Recommended because of some alignment with you...,Ella Rock was recommended because it has a sem...
3,U001,D054,Mount Lavinia Beach,beach,5000,59,31.827333,Low,0.528928,0.627124,0.558387,4,Recommended because of some alignment with you...,Mount Lavinia Beach was recommended because it...
4,U001,D053,Dutch Hospital Colombo,heritage;city,5000,60,33.808425,Medium,0.589435,0.450518,0.547760,5,Recommended because of good match with your tr...,Dutch Hospital Colombo was recommended because...
5,U001,D020,Nuwara Eliya City,city;nature,5000,67,33.082968,Medium,0.527826,0.515189,0.524035,6,Recommended because of some alignment with you...,Nuwara Eliya City was recommended because it h...
6,U001,D066,Bopath Ella,nature,3000,57,33.713491,Medium,0.541864,0.458981,0.516999,7,Recommended because of good match with your tr...,Bopath Ella was recommended because it has a s...
7,U001,D098,Ohiya,nature;adventure,5000,63,35.161483,Medium,0.557703,0.329898,0.489361,8,Recommended because of good match with your tr...,Ohiya was recommended because it has a semanti...
8,U001,D070,Brief Garden,nature,4000,59,35.603315,Medium,0.532182,0.290510,0.459681,9,Recommended because of some alignment with you...,Brief Garden was recommended because it has a ...
9,U001,D052,Independence Square,heritage;city,2000,60,36.897572,Medium,0.552463,0.175132,0.439264,10,Recommended because of good match with your tr...,Independence Square was recommended because it...


In [13]:
sample_user = final_explainable[
    "user_id"
].iloc[0]

print("Sample user:", sample_user)

display(
    final_explainable[
        final_explainable["user_id"] == sample_user
    ][
        [
            "demand_aware_rank",
            "name",
            "category",
            "predicted_crowd_level",
            "demand_aware_score",
            "explanation"
        ]
    ].sort_values(
        "demand_aware_rank"
    )
)

Sample user: U001


,demand_aware_rank,name,category,predicted_crowd_level,demand_aware_score,explanation
0,1,Colombo National Museum,heritage;culture,Low,0.638371,Recommended because of good match with your tr...
1,2,Viharamahadevi Park,nature;city,Low,0.614931,Recommended because of good match with your tr...
2,3,Ella Rock,nature;adventure,Low,0.588797,Recommended because of some alignment with you...
3,4,Mount Lavinia Beach,beach,Low,0.558387,Recommended because of some alignment with you...
4,5,Dutch Hospital Colombo,heritage;city,Medium,0.547760,Recommended because of good match with your tr...
5,6,Nuwara Eliya City,city;nature,Medium,0.524035,Recommended because of some alignment with you...
6,7,Bopath Ella,nature,Medium,0.516999,Recommended because of good match with your tr...
7,8,Ohiya,nature;adventure,Medium,0.489361,Recommended because of good match with your tr...
8,9,Brief Garden,nature,Medium,0.459681,Recommended because of some alignment with you...
9,10,Independence Square,heritage;city,Medium,0.439264,Recommended because of good match with your tr...


In [14]:
explainable_path = (
    RESULTS_DIR /
    "explainable_recommendations.csv"
)

final_explainable.to_csv(
    explainable_path,
    index=False
)

print("Saved successfully.")
print("Path:", explainable_path)
print("Exists:", explainable_path.exists())

Saved successfully.
Path: ..\results\explainable_recommendations.csv
Exists: True


In [15]:
print("========== NOTEBOOK 08 VALIDATION ==========")

assert len(final_explainable) == 500

assert (
    final_explainable["user_id"].nunique()
    == 50
)

assert (
    final_explainable["demand_aware_rank"].min()
    == 1
)

assert (
    final_explainable["demand_aware_rank"].max()
    == 10
)

assert (
    final_explainable["explanation"]
    .notna()
    .all()
)

assert (
    final_explainable["detailed_explanation"]
    .notna()
    .all()
)

assert explainable_path.exists()

print("Rows:", len(final_explainable))
print("Users:", final_explainable["user_id"].nunique())
print("Explanations generated: ✓")
print("Detailed explanations generated: ✓")
print("Output saved: ✓")

print("\n✓ NOTEBOOK 08 COMPLETED SUCCESSFULLY")

========== NOTEBOOK 08 VALIDATION ==========
Rows: 500
Users: 50
Explanations generated: ✓
Detailed explanations generated: ✓
Output saved: ✓

✓ NOTEBOOK 08 COMPLETED SUCCESSFULLY
